# Assignment 2: E-Commerce Data Analytics & Metrics

Assignment 1 mein data clean ho gaya. Ab Assignment 2 mein hum **meaningful business insights (analytics)** nikalenge:
1. **Average Rating**: Sabhi users ki average rating calculate karna
2. **Percentage of Poor Ratings**: Poor rating (e.g. rating < 3.0) dene wale users ka percentage nikalna

## Step 0: Data Load & Clean (From Assignment 1)
Pehle hum [store_data.json](store_data.json) ko load karke Assignment 1 ke pipeline se clean kar lete hain taaki clean data par analysis ho.

In [1]:
import json

# 1. Load Data
with open("store_data.json", "r") as f:
    raw_data = json.load(f)

# 2. Rating cleaner mapper
word_to_num = {"one": 1.0, "two": 2.0, "three": 3.0, "four": 4.0, "five": 5.0}

def clean_rating(val):
    if val is None: return None
    s = str(val).strip().lower()
    if s in word_to_num: return word_to_num[s]
    try: return float(s)
    except ValueError: return None

# 3. Clean and Deduplicate
cleaned_data = []
seen = set()
for item in raw_data:
    name = item.get("name", "").strip().capitalize()
    raw_age = item.get("age")
    age = int(raw_age) if (raw_age and str(raw_age).strip().isdigit()) else None
    
    unique_key = (name.lower(), age)
    if unique_key in seen:
        continue
    seen.add(unique_key)
    
    raw_rating = item.get("ratng") if "ratng" in item else item.get("rating")
    rating = clean_rating(raw_rating)
    feedback = item.get("feedback", "").strip()
    
    cleaned_data.append({"name": name, "rating": rating, "feedback": feedback, "age": age})

print(f"Total Clean Records: {len(cleaned_data)}")
for r in cleaned_data:
    print(r)

Total Clean Records: 5
{'name': 'Alice', 'rating': 5.0, 'feedback': 'great product!!', 'age': 25}
{'name': 'Bob', 'rating': 4.0, 'feedback': 'ok but late delivery', 'age': 30}
{'name': 'Diana', 'rating': 5.0, 'feedback': 'love it', 'age': 28}
{'name': 'Rahul', 'rating': 2.0, 'feedback': 'bad experiance', 'age': None}
{'name': 'Eve', 'rating': 3.5, 'feedback': 'average, not bad', 'age': 35}


## Point 1: Average Rating Nikalna
- **Concept**: Sabhi users ki rating ka sum / Total users ki count.
- **Edge Case / System Design Check**:
  - Agar kisi user ki rating `None` ho toh usko sum/count me consider nahi karna chahiye taaki average distort na ho.
  - Agar dataset empty ho toh `ZeroDivisionError` se bachne ke liye safe check lagana zaroori hai.

In [2]:
def calculate_average_rating(records):
    # Sirf un ratings ko filter karo jo valid numbers hain (not None)
    valid_ratings = [r["rating"] for r in records if r.get("rating") is not None]
    
    if not valid_ratings:
        return 0.0
    
    total_sum = sum(valid_ratings)
    count = len(valid_ratings)
    avg = total_sum / count
    
    return round(avg, 2)

avg_rating = calculate_average_rating(cleaned_data)
print(f"Average Rating of Store: {avg_rating} ⭐ (out of 5.0)")

Average Rating of Store: 3.9 ⭐ (out of 5.0)


## Point 2: Percentage of Users with Poor Rating
- **Concept**: Business ke liye yeh jan na critical hai ki kitne percent users asantusht (unhappy) hain.
- **Rule**: E-commerce standard ke mutabiq, rating `< 3.0` (yaani 1 ya 2 stars) ko **Poor Rating** mana jata hai.
- **Formula**: 
  $$\text{Poor Rating \%} = \left(\frac{\text{Poor Rating dene wale users}}{\text{Total Valid Users}}\right) \times 100$$

In [3]:
def calculate_poor_rating_percentage(records, threshold=3.0):
    valid_records = [r for r in records if r.get("rating") is not None]
    
    if not valid_records:

        return 0.0, []
    
    # Poor ratings filter karte hain (threshold se kam, i.e., < 3.0)
    poor_users = [r for r in valid_records if r["rating"] < threshold]
    
    poor_count = len(poor_users)
    total_count = len(valid_records)
    
    percentage = (poor_count / total_count) * 100
    
    return round(percentage, 2), poor_users

poor_pct, unhappy_customers = calculate_poor_rating_percentage(cleaned_data)

print(f"Percentage of Users with Poor Rating (< 3.0): {poor_pct}%")
print(f"\nUnhappy Customers ({len(unhappy_customers)}): ")
for u in unhappy_customers:
    print(f"- {u['name']}: {u['rating']} stars (Feedback: '{u['feedback']}')")

Percentage of Users with Poor Rating (< 3.0): 20.0%

Unhappy Customers (1): 
- Rahul: 2.0 stars (Feedback: 'bad experiance')


## 📊 Final Analytics Summary Dashboard

In [4]:
print("=" * 45)
print("      E-COMMERCE METRICS DASHBOARD")
print("=" * 45)
print(f" Total Cleaned Customers : {len(cleaned_data)}")
print(f" Average Store Rating     : {avg_rating} / 5.0 ⭐")
print(f" Poor Rating Percentage   : {poor_pct}%")
print(f" Happy/Satisfied Users    : {round(100 - poor_pct, 2)}%")
print("=" * 45)

      E-COMMERCE METRICS DASHBOARD
 Total Cleaned Customers : 5
 Average Store Rating     : 3.9 / 5.0 ⭐
 Poor Rating Percentage   : 20.0%
 Happy/Satisfied Users    : 80.0%
